# CR-HSDPA r=0.50 Training - Epochs 101 \u2013 150 (101\u2013150 of 200)
**Thesis Topic:** A Modified Attention-PestNet with Channel-Reduced Hierarchical Scaled Dot-Product Attention for Efficient Insect Pest Detection  
**Authors:** Lean Adrian Murillo, James Oliver C. Mendoza, DM Rashid P. Ferrer, Charisse P. Barbosa (University of Mindanao)  
**Target Architecture:** CR-HSDPA (`cfg/cr_hsdpa_050.yaml`, r=0.50)  
**Dataset:** IP102-YOLO (`dmrashidpferrer/ip102-yolo-dataset`)  
**Training Stage:** Epochs 101 \u2013 150 (`lr0=0.003`, SGD, batch=32, imgsz=640, Dual T4)  
**Checkpointing:** `epochs=50` per stage, `save_period=10` + auto `best.pt/last.pt`. Publish stage output as Kaggle dataset for next stage.  
**Branch:** `HSDPA-DM-Testing` (reuses `90377Sednaaa/HSDPA`, account-agnostic via Kaggle CLI).  
---
Run cells top-to-bottom exactly once (Rule 4: single `model.train()` cell). After run, save output version and attach it to the next stage kernel.


In [ ]:
# 1. Setup repository and install ultralytics in editable mode
# Clones 90377Sednaaa/HSDPA, preferring HSDPA-DM-Testing but falling back
# to the default branch (training code is identical there; the branch only
# adds kaggle/ notebooks, which the kernel already carries).
import os
os.environ["WANDB_MODE"] = "disabled"
import subprocess
import sys

REPO_URL = "https://github.com/90377Sednaaa/HSDPA.git"
BRANCH = "HSDPA-DM-Testing"
REPO_DIR = "/kaggle/working/HSDPA"

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout or "") + (r.stderr or "")
    if out.strip():
        print(out[-3000:])
    return r.returncode

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL} (branch {BRANCH}) ...")
    if sh(f"git clone -b {BRANCH} {REPO_URL} {REPO_DIR}") != 0:
        print(f"Branch '{BRANCH}' not found upstream - falling back to default branch")
        if sh(f"git clone {REPO_URL} {REPO_DIR}") != 0:
            raise SystemExit("git clone failed: check internet access and repo URL")

os.chdir(REPO_DIR)
if os.path.exists(os.path.join(REPO_DIR, ".git")):
    sh("git fetch origin")
    if sh(f"git checkout {BRANCH}") != 0:
        print(f"Branch '{BRANCH}' not on remote - staying on default branch")
    else:
        sh(f"git pull --ff-only origin {BRANCH}")

if not os.path.exists(os.path.join(REPO_DIR, "setup.py")):
    raise SystemExit(f"Repo checkout broken: no setup.py in {REPO_DIR}")
if sh("pip install -q -e .") != 0:
    raise SystemExit("pip install -e . failed")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
import ultralytics
print("ultralytics import OK")
print("Environment ready at:", REPO_DIR)


In [ ]:
# 2. Check GPU environment and CUDA capabilities
import torch
from ultralytics import YOLO

print("PyTorch version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
device_count = torch.cuda.device_count()
print("GPU Count:", device_count)
for i in range(device_count):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")


In [ ]:
# 3. Verify IP102-YOLO dataset directory and images
# Attach in Kaggle UI or CLI: dmrashidpferrer/ip102-yolo-dataset
import os, glob, yaml

DATA_ROOT = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "images" in dirs and os.path.exists(os.path.join(root, "images/train")):
        DATA_ROOT = root
        break

if not DATA_ROOT:
    raise FileNotFoundError("Could not locate IP102 dataset in /kaggle/input. Attach dmrashidpferrer/ip102-yolo-dataset.")

print(f"Dataset root verified at: {DATA_ROOT}")
train_count = len(os.listdir(os.path.join(DATA_ROOT, "images/train")))
val_count = len(os.listdir(os.path.join(DATA_ROOT, "images/val")))
print(f"Train images: {train_count} (Expected: 15,178)")
print(f"Validation images: {val_count} (Expected: 3,798)")

with open("cfg/ip102.yaml", "r") as f:
    ip102_cfg = yaml.safe_load(f)
ip102_cfg["path"] = DATA_ROOT
with open("cfg/ip102.yaml", "w") as f:
    yaml.dump(ip102_cfg, f, default_flow_style=False, sort_keys=False)
print("cfg/ip102.yaml dynamically configured and ready!")


In [ ]:
# 4. Train CR-HSDPA r=0.50 (Stage 3: continue to {50*stage} cumulative epochs)
# Attach prior stage output dataset (e.g. cr-r050-stage{prev} output) as Kaggle input.
import torch
import os, glob, time
os.environ["WANDB_MODE"] = "disabled"
from ultralytics import YOLO

MODEL_CFG = "cfg/cr_hsdpa_050.yaml"
PROJECT_NAME = "dataset_r050_stage3"
EXP_NAME = "ip102_cr050_stage3"
PREV_EXP = "ip102_cr050_stage2"

# 1) Prefer prior-stage best.pt from /kaggle/input (cross-stage continuation)
candidate_ckpts = glob.glob("/kaggle/input/**/best.pt", recursive=True)
# 2) Fallback: interrupted THIS stage (same project/name)
local_last = glob.glob(f"{PROJECT_NAME}/{EXP_NAME}/weights/last.pt")

resume_flag = False
if local_last and not candidate_ckpts:
    print(f"Resuming interrupted Stage {stage}: {local_last[0]}")
    model = YOLO(local_last[0])
    resume_flag = True
elif candidate_ckpts:
    print(f"=== Continuing Stage 3 from prior-stage weights: {candidate_ckpts[0]} ===")
    model = YOLO(candidate_ckpts[0])
    resume_flag = False  # new save_dir -> must NOT use resume=True
else:
    print(f"Warning: prior checkpoint not found. Falling back to {MODEL_CFG}")
    model = YOLO(MODEL_CFG)
    resume_flag = False

# Multi-GPU support for dual T4
device_arg = [0, 1] if torch.cuda.device_count() >= 2 else 0

train_start_wall = time.time()

results = model.train(
    data="cfg/ip102.yaml",
    epochs=50,
    batch=32,
    imgsz=640,
    device=device_arg,
    optimizer="SGD",
    lr0=0.003,
    momentum=0.9,
    amp=True,
    save_period=10,
    project=PROJECT_NAME,
    name=EXP_NAME,
    exist_ok=True,
    workers=4,
    resume=resume_flag,
    verbose=True
)

train_elapsed_sec = time.time() - train_start_wall
train_elapsed_hrs = train_elapsed_sec / 3600.0
print(f"\nStage 3 CR-HSDPA r=0.50 training completed in {train_elapsed_hrs:.2f} hours ({train_elapsed_sec:.1f} s).")
os.makedirs(f"{PROJECT_NAME}/{EXP_NAME}", exist_ok=True)
with open(f"{PROJECT_NAME}/{EXP_NAME}/training_time.txt", "w") as f:
    f.write(f"{train_elapsed_sec:.2f}\n")


In [ ]:
# 5. Comprehensive Thesis Evaluation & Efficiency Benchmark (Section 2.7 Protocol)
import time, os, glob
import torch
import pandas as pd
from ultralytics import YOLO

best_ckpt = f"{PROJECT_NAME}/{EXP_NAME}/weights/best.pt"
if os.path.exists(best_ckpt):
    print(f"=== Evaluating CR-HSDPA r=0.50 Stage 3 Best Checkpoint: {best_ckpt} ===\n")
    eval_model = YOLO(best_ckpt)

    # 1. Detection Performance (mAP@50, mAP@50:95, Precision, Recall)
    val_results = eval_model.val(data="cfg/ip102.yaml", split="val", batch=32, imgsz=640, device=0)
    p = val_results.box.mp
    r = val_results.box.mr
    map50 = val_results.box.map50
    map50_95 = val_results.box.map

    # 2. Computational Efficiency Benchmark (Section 2.7: FP32, batch 1, 20 warmup, 200 timed runs)
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    bench_model = eval_model.model.to(device).float()
    bench_model.eval()
    dummy_input = torch.randn(1, 3, 640, 640, device=device)

    with torch.no_grad():
        for _ in range(20):
            _ = bench_model(dummy_input)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start_time = time.perf_counter()
        for _ in range(200):
            _ = bench_model(dummy_input)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
        total_time = time.perf_counter() - start_time

    latency_ms = (total_time / 200.0) * 1000.0
    fps = 200.0 / total_time
    total_params = sum(x.numel() for x in eval_model.model.parameters())
    model_size_mb = os.path.getsize(best_ckpt) / (1024 * 1024)

    time_file = f"{PROJECT_NAME}/{EXP_NAME}/training_time.txt"
    train_time_hrs = None
    if os.path.exists(time_file):
        with open(time_file, "r") as f:
            train_time_hrs = float(f.read().strip()) / 3600.0

    print("\n" + "=" * 65)
    print("   THESIS EVALUATION METRICS SUMMARY - CR-HSDPA r=0.50 STAGE 3 (Section 2.7)")
    print("=" * 65)
    print(f"  Precision (P):          {{p:.4f}} ({{p*100:.2f}}%)")
    print(f"  Recall (R):             {{r:.4f}} ({{r*100:.2f}}%)")
    print(f"  mAP@50:                 {{map50:.4f}} ({{map50*100:.2f}}%)")
    print(f"  mAP@50:95:              {{map50_95:.4f}} ({{map50_95*100:.2f}}%)")
    print(f"  Parameters:             {{total_params:,}}")
    if train_time_hrs:
        print(f"  Training Time:          {{train_time_hrs:.2f}} hours (Stage 3)")
    print(f"  Inference Latency:      {{latency_ms:.2f}} ms")
    print(f"  Throughput (FPS):       {{fps:.2f}} FPS")
    print(f"  Model File Size:        {{model_size_mb:.2f}} MB")
    print("=" * 65)

    summary_df = pd.DataFrame([{{
        "Model": "CR-HSDPA r=0.50 (Stage 3)",
        "Precision": p,
        "Recall": r,
        "mAP@50": map50,
        "mAP@50-95": map50_95,
        "Parameters": total_params,
        "Training_Time_Hours": round(train_time_hrs, 2) if train_time_hrs else "N/A",
        "Latency_ms": latency_ms,
        "FPS": fps,
        "ModelSize_MB": model_size_mb
    }}])
    out_csv = f"{PROJECT_NAME}/{EXP_NAME}/thesis_metrics_summary_cr050_stage3.csv"
    summary_df.to_csv(out_csv, index=False)
    print(f"Saved Stage 3 metrics summary to: {{out_csv}}")
else:
    print(f"Checkpoint {{best_ckpt}} not found.")
